# GestureX data collection and schema inspection

This notebook accompanies `src/collect_data.py`; it does not fabricate a dataset or try to hide missing samples. Collect data with the webcam script first, then use this notebook to inspect the saved schema and draw one real landmark sample.

For a defensible generalization experiment, record separate training, validation, and held-out cross-session sessions. The final session should differ in lighting, camera position, hand orientation, background, or subject.

## 1. Collect local samples

From the repository root, activate the project environment and run:

```bash
python src/collect_data.py
```

The collector asks for a supported gesture, a session ID, and a subject ID, shows progress, and writes a row only after it has a valid 21-landmark hand detection. Use descriptive IDs such as `train_s01`, `train_s02`, and `test_s01`; do not reuse the held-out test ID during model selection.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd


def find_repository_root(start: Path) -> Path:
    """Find the repository whether Jupyter started at root or notebooks/."""
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'config.py').is_file():
            return candidate
    raise RuntimeError(
        'Could not find the GestureX repository root. Start Jupyter from the repository.'
    )


ROOT = find_repository_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import METADATA_COLUMNS, RAW_DATA_DIR, RAW_FEATURE_COLUMNS
from src.landmarks import HAND_CONNECTIONS, unflatten_landmarks

ROOT, RAW_DATA_DIR


## 2. Load the collected CSV

The canonical table stores `sample_id`, `session_id`, `subject_id`, `gesture`, and `timestamp`, followed by 63 coordinates. Coordinate ordering is exactly `(landmark_0_x, landmark_0_y, landmark_0_z, ..., landmark_20_z)`. Landmark 0 is the wrist.

In [ ]:
csv_paths = sorted(RAW_DATA_DIR.glob('*.csv'))
if not csv_paths:
    raise FileNotFoundError(
        f'No collected CSV found in {RAW_DATA_DIR}. Run python src/collect_data.py first.'
    )

frames = [pd.read_csv(path) for path in csv_paths]
data = pd.concat(frames, ignore_index=True)
required_columns = [*METADATA_COLUMNS, *RAW_FEATURE_COLUMNS]
missing_columns = [column for column in required_columns if column not in data.columns]
if missing_columns:
    raise ValueError(
        'Collected CSV does not match the GestureX schema. Missing columns: ' +
        ', '.join(missing_columns)
    )
if data.empty:
    raise ValueError('The collected CSV has no valid samples yet.')

print(f'Loaded {len(data):,} samples from {len(csv_paths)} CSV file(s).')
data.head()


## 3. Inspect the schema and session coverage

The following tables expose the actual types and coverage. A missing label or an unexpected session should be corrected before training—not silently patched later.

In [ ]:
schema = pd.DataFrame({
    'column': data.columns,
    'dtype': [str(data[column].dtype) for column in data.columns],
    'missing_values': [int(data[column].isna().sum()) for column in data.columns],
})
display(schema)

coverage = (
    data.groupby(['session_id', 'subject_id', 'gesture'], dropna=False)
    .size()
    .rename('samples')
    .reset_index()
    .sort_values(['session_id', 'gesture'])
)
coverage


## 4. Draw one real MediaPipe landmark sample

This plot uses the first stored row and the MediaPipe hand topology. The vertical axis is inverted to match image coordinates. It is an inspection aid, not a claimed model result.

In [ ]:
sample = data.iloc[0]
points = unflatten_landmarks(
    sample.loc[list(RAW_FEATURE_COLUMNS)].to_numpy(dtype=float)
)

fig, ax = plt.subplots(figsize=(6, 6))
for start, end in HAND_CONNECTIONS:
    ax.plot(points[[start, end], 0], points[[start, end], 1], color='tab:blue', lw=1.5)
scatter = ax.scatter(points[:, 0], points[:, 1], c=points[:, 2], cmap='viridis', s=55, zorder=2)
for index, (x_coord, y_coord, _) in enumerate(points):
    ax.annotate(str(index), (x_coord, y_coord), xytext=(3, 3), textcoords='offset points', fontsize=8)
ax.set_aspect('equal')
ax.invert_yaxis()
ax.set_xlabel('Normalized x')
ax.set_ylabel('Normalized y')
ax.set_title(
    f"Stored landmark sample: {sample['gesture']} | session={sample['session_id']}"
)
fig.colorbar(scatter, ax=ax, label='Normalized z')
plt.show()


## Collection quality checklist

- Confirm each configured gesture has enough samples in every planned session.
- Verify that `session_id` separates recording events rather than merely naming random row subsets.
- Keep the final cross-session session untouched until model selection is complete.
- Do not commit webcam recordings or collected participant data without consent; the repository ignores these files by default.